# Autotrading Experiment Analysis

Analysis of autonomous trading strategy optimization results from `results.tsv`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_csv("results.tsv", sep="\t")
df["composite_score"] = pd.to_numeric(df["composite_score"], errors="coerce")
df["sharpe"] = pd.to_numeric(df["sharpe"], errors="coerce")
df["max_dd"] = pd.to_numeric(df["max_dd"], errors="coerce")
df["status"] = df["status"].str.strip().str.upper()

print(f"Total experiments: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head(10)

In [ ]:
counts = df["status"].value_counts()
print("Experiment outcomes:")
print(counts.to_string())

n_keep = counts.get("KEEP", 0)
n_discard = counts.get("DISCARD", 0)
n_crash = counts.get("CRASH", 0)
n_decided = n_keep + n_discard
if n_decided > 0:
    print(f"\nKeep rate: {n_keep}/{n_decided} = {n_keep / n_decided:.1%}")

In [ ]:
kept = df[df["status"] == "KEEP"].copy()
print(f"KEPT experiments ({len(kept)} total):\n")
for i, row in kept.iterrows():
    score = row["composite_score"]
    sharpe = row["sharpe"]
    dd = row["max_dd"]
    desc = row["description"]
    print(f"  #{i:3d}  score={score:.4f}  sharpe={sharpe:.2f}  dd={dd:.3f}  {desc}")

## Composite Score Over Time

Track how the best composite_score evolves. The running maximum shows the frontier.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 12), gridspec_kw={"height_ratios": [2, 1]})

# --- Top: Composite Score ---
ax = axes[0]
valid = df[df["status"] != "CRASH"].copy().reset_index(drop=True)

disc = valid[valid["status"] == "DISCARD"]
ax.scatter(disc.index, disc["composite_score"], c="#cccccc", s=12, alpha=0.5, zorder=2, label="Discarded")

kept_v = valid[valid["status"] == "KEEP"]
ax.scatter(kept_v.index, kept_v["composite_score"], c="#2ecc71", s=50, zorder=4, label="Kept", edgecolors="black", linewidths=0.5)

kept_mask = valid["status"] == "KEEP"
kept_idx = valid.index[kept_mask]
kept_scores = valid.loc[kept_mask, "composite_score"]
running_max = kept_scores.cummax()
ax.step(kept_idx, running_max, where="post", color="#27ae60", linewidth=2, alpha=0.7, zorder=3, label="Running best")

for idx, score in zip(kept_idx, kept_scores):
    desc = str(valid.loc[idx, "description"]).strip()
    if len(desc) > 40: desc = desc[:37] + "..."
    ax.annotate(desc, (idx, score), textcoords="offset points", xytext=(6, 6), fontsize=7.5, color="#1a7a3a", alpha=0.9, rotation=25, ha="left", va="bottom")

ax.set_xlabel("Experiment #")
ax.set_ylabel("Composite Score (higher is better)")
ax.set_title(f"Autotrading Progress: {len(df)} Experiments, {len(kept_v)} Kept")
ax.legend(loc="lower right")
ax.grid(True, alpha=0.2)
ax.axhline(y=0, color="red", linestyle="--", alpha=0.3, label="Break-even")

# --- Bottom: Sharpe and Max Drawdown ---
ax2 = axes[1]
kept_df = df[df["status"] == "KEEP"].reset_index()
x = range(len(kept_df))
ax2.bar(x, kept_df["sharpe"], alpha=0.6, color="#3498db", label="Sharpe")
ax2_twin = ax2.twinx()
ax2_twin.plot(x, kept_df["max_dd"], "ro-", markersize=5, alpha=0.7, label="Max DD")
ax2.set_xlabel("Kept Experiment #")
ax2.set_ylabel("Sharpe Ratio", color="#3498db")
ax2_twin.set_ylabel("Max Drawdown", color="red")
ax2.legend(loc="upper left")
ax2_twin.legend(loc="upper right")
ax2.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig("progress.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to progress.png")

## Summary Statistics

In [ ]:
kept = df[df["status"] == "KEEP"].copy()
baseline = df.iloc[0]["composite_score"]
best_score = kept["composite_score"].max()
best_row = kept.loc[kept["composite_score"].idxmax()]

print(f"Baseline score:    {baseline:.6f}")
print(f"Best score:        {best_score:.6f}")
print(f"Total improvement: {best_score - baseline:.6f}")
print(f"Best Sharpe:       {best_row['sharpe']:.4f}")
print(f"Best Max DD:       {best_row['max_dd']:.4f}")
print(f"Best experiment:   {best_row['description']}")
print()
print("Improvement history:")
for i, (_, row) in enumerate(kept.iterrows()):
    print(f"  #{i}: score={row['composite_score']:.4f}  sharpe={row['sharpe']:.2f}  dd={row['max_dd']:.3f}  {row['description']}")

## Top Improvements (Ranked by Delta)

In [ ]:
kept = df[df["status"] == "KEEP"].copy()
kept["prev_score"] = kept["composite_score"].shift(1)
kept["delta"] = kept["composite_score"] - kept["prev_score"]
hits = kept.iloc[1:].copy()
hits = hits.sort_values("delta", ascending=False)

print(f"{'Rank':>4}  {'Delta':>8}  {'Score':>8}  {'Sharpe':>7}  {'MaxDD':>6}  Description")
print("-" * 90)
for rank, (_, row) in enumerate(hits.iterrows(), 1):
    print(f"{rank:4d}  {row['delta']:+.4f}  {row['composite_score']:.4f}  {row['sharpe']:.2f}     {row['max_dd']:.3f}  {row['description']}")
print(f"\n{'':>4}  {hits['delta'].sum():+.4f}  {'':>8}  TOTAL improvement")